# Direct 2D N-body acceleration (research prototype)

Compute one softened-gravity acceleration evaluation for `n` unit-mass bodies in the
plane. Body `i` sits at `(px[i], py[i])` (a fixed deterministic fill, so the result is
reproducible); its acceleration is the direct sum over every other body `j`:
`ax_i = sum_{j != i} (px_j - px_i) / d3` and likewise `ay_i`, with
`d3 = (r2 + EPS2) * sqrt(r2 + EPS2)`, `r2 = dx*dx + dy*dy` and `EPS2 = 0.0025`
(softening 0.05 squared). We report the checksum `sum_i (|ax_i| + |ay_i|)` as the
oracle -- the absolute values matter, because the raw accelerations nearly cancel in
the sum. The C port must implement the same direct O(n^2) sum and match this checksum.

In [ ]:
N_TEST = 1024
EPS2 = 0.0025          # softening 0.05 squared

# deterministic positions in [-1, 1): exact integer arithmetic, then one division,
# so C and Python produce bit-identical doubles
def px_of(i):
    return ((i * 73) % 4093) / 4093.0 * 2.0 - 1.0

def py_of(i):
    return ((i * 151) % 4093) / 4093.0 * 2.0 - 1.0

px = [px_of(i) for i in range(N_TEST)]
py = [py_of(i) for i in range(N_TEST)]

In [ ]:
import math

# direct O(n^2) acceleration sum: every body attracts every other, unit masses
def nbody_accel(px, py, n):
    ax = [0.0] * n
    ay = [0.0] * n
    for i in range(n):
        sx = 0.0
        sy = 0.0
        for j in range(n):
            if j == i:
                continue
            dx = px[j] - px[i]
            dy = py[j] - py[i]
            r2 = dx * dx + dy * dy
            d3 = (r2 + EPS2) * math.sqrt(r2 + EPS2)
            sx += dx / d3
            sy += dy / d3
        ax[i] = sx
        ay[i] = sy
    return ax, ay

ax, ay = nbody_accel(px, py, N_TEST)
terms = [abs(ax[i]) + abs(ay[i]) for i in range(N_TEST)]
checksum = sum(terms)
print('checksum =', checksum)   # -> 1053395.4952812886

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any
# language) must reproduce this checksum within tol. The per-body values
# t_i = |ax_i| + |ay_i| are handed over as the terms of the checksum -- the
# absolute values avoid the near-total cancellation that makes the raw sum of
# the accelerations ~0 -- so the extraction tool can measure how much the sum
# legitimately moves under reordering.
ORACLE = {'kernel': 'nbody',
          'params': {'N': N_TEST},
          'expected': {'CHECKSUM': checksum},
          'terms': {'CHECKSUM': terms},
          'tol': 1e-6}